In [1]:
import os
import sys
import json

import pandas as pd
import numpy as np

from openai import AsyncOpenAI

from dotenv import load_dotenv
load_dotenv()

base_path = os.path.dirname(os.getcwd()) 
if base_path not in sys.path:
    sys.path.append(base_path)

from src.prompts import SYSTEM_MSG, TEMPLATE, INSTRUCTION, INPUT_STR
from src.models.trend_analysis import TA_Model
from src.models.sentiment_analysis import SA_Model

/Users/juhyeongpang/Desktop/Projects/Quant/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to /Users/juhyeongpang/Desktop
[nltk_data]     /Projects/Quant/src/utils/../../.venv/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/juhyeongpang/Desktop
[nltk_data]     /Projects/Quant/src/utils/../../.venv/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to /Use
[nltk_data]     rs/juhyeongpang/Desktop/Projects/Quant/src/utils/../..
[nltk_data]     /.venv/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /Users/juhyeongpa

In [2]:
ta_model = TA_Model()
sa_model = SA_Model()

Augmenting 'negative' class: 604 -> 2000...
Augmenting 'positive' class: 1363 -> 2000...


In [ ]:
trend_prediction_raw = ta_model.predict("AAPL")["Prediction"]
trend_prediction_raw = round(trend_prediction_raw, 2)
trend_prediction = "+" + str(trend_prediction_raw) if (trend_prediction_raw >= 0) else str(trend_prediction_raw)

print(trend_prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
-0.04197556525468826


In [6]:
sentiment_prediction_raw = sa_model.predict("GOOG")
positive_sen_ratio = round(sentiment_prediction_raw['Positive'], 2)
neutral_sen_ratio = round(sentiment_prediction_raw['Neutral'], 2)
negative_sen_ratio = round(sentiment_prediction_raw['Negative'], 2)

In [7]:
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [8]:

lstm_output = f"LSTM model output : {trend_prediction}%"
bert_output = f"BERT model output : {positive_sen_ratio}% Positive, {neutral_sen_ratio}% Neutral, {negative_sen_ratio}% Negative"
macro_data = "Inflation Rate Higher than normal"
open_price = 312
cash = 30043
shares_owned = 30

input_str = INPUT_STR.format(
    lstm_output=lstm_output,
    bert_output=bert_output,
    macro_data=macro_data,
    open_price=open_price,
    cash=cash,
    shares_owned=shares_owned
)

instruction = INSTRUCTION.format(
    input_str=input_str,
)

prompt = TEMPLATE.format(
    instruction=instruction,
)

response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_MSG},
                {"role": "user", "content": prompt},
            ],
            temperature=0.3,
            response_format={"type": "json_object"},
        )

print(response)

ChatCompletion(id='chatcmpl-DNsUYlh9pX5Te0SLs3OmSmsmxtQXI', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n    "action": "SELL",\n    "share": "30",\n    "reason": "The LSTM model indicates a negative return, and the sentiment analysis shows a lack of strong positive sentiment."\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1774583382, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_37b32d27ed', usage=CompletionUsage(completion_tokens=42, prompt_tokens=353, total_tokens=395, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [9]:
result_json = json.loads(response.choices[0].message.content)

In [10]:
result_json

{'action': 'SELL',
 'share': '30',
 'reason': 'The LSTM model indicates a negative return, and the sentiment analysis shows a lack of strong positive sentiment.'}

In [12]:
action = result_json.get("action", "SELL")
share = result_json.get("share", "30")
reason = result_json.get("reason", "-")

In [11]:
print(prompt)


This is the INPUT you should refer to:

LSTM Prediction: LSTM model output : -0.04197556525468826%
BERT Sentiment: BERT model output : 0.14% Positive, 0.75% Neutral, 0.11% Negative
Opening Price: $312
Cash: $30043
Current Holdings: 30 shares


You must provide a concise decision.


Decisions:


In [14]:
share

'30'